# LSTM baseline results

Summarises the English LSTM baseline runs in `results/lstm_baseline/runs` (three tasks x three seeds). Run with the project kernel: `uv run jupyter lab`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS = ROOT / "results" / "lstm_baseline" / "runs"

In [ ]:
rows = []
for path in sorted(RUNS.glob("*/metrics.json")):
    m = json.loads(path.read_text())
    for model in ("lstm_test", "majority_test"):
        rows.append({"run": path.parent.name, "task": m["config"]["task"], "seed": m["config"]["seed"],
                     "model": model.removesuffix("_test"),
                     **{k: m[model][k] for k in ("exact_match_accuracy", "micro_f1", "macro_f1")}})
metrics = pd.DataFrame(rows)
metrics.groupby(["task", "model"])[["exact_match_accuracy", "micro_f1", "macro_f1"]].agg(["mean", "std"]).round(4)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, task in zip(axes, ["fine", "coarse", "parent"]):
    for path in sorted(RUNS.glob(f"english_{task}_seed*/history.csv")):
        h = pd.read_csv(path)
        ax.plot(h["epoch"], h["validation_bce"], label=path.parent.name.rsplit("_", 1)[1])
    ax.set(title=task, xlabel="epoch")
    ax.legend()
axes[0].set_ylabel("validation BCE")
plt.tight_layout()

## Using the package from a notebook

The `src/` methods are installed in editable mode, so they import directly:

In [ ]:
from lstm_baseline.prepare_etymology import FINE_LABELS, COARSE_LABELS
FINE_LABELS, COARSE_LABELS